In [8]:
import os
import random
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

DATA_DIR = "../data/preprocessed"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
VAL_CSV   = os.path.join(DATA_DIR, "val.csv")
TEST_CSV  = os.path.join(DATA_DIR, "test.csv")

SEED = 42
BATCH_SIZE = 64
LR = 1e-3
EPOCHS = 100
HIDDEN = [64, 32]
DROPOUT = 0.1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

Device: cpu


In [9]:
# Hàm đọc (giả sử target là cột cuối)
def read_xy(path):
    df = pd.read_csv(path)
    df = df.dropna(how="all")
    X = df.iloc[:, :-1].values.astype(np.float32)
    y = df.iloc[:, -1].values.astype(np.float32).reshape(-1, 1)
    return df, X, y

df_train, X_train, y_train = read_xy(TRAIN_CSV)
df_val,   X_val,   y_val   = read_xy(VAL_CSV)
df_test,  X_test,  y_test  = read_xy(TEST_CSV)

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)
display(df_train.head())

Train: (400, 5) (400, 1)
Val:   (50, 5) (50, 1)
Test:  (50, 5) (50, 1)


,gender,age,annual Salary,credit card debt,net worth,car purchase amount
0,0.0,0.325146,0.629180,0.488604,0.622060,0.523032
1,1.0,0.399740,0.451081,0.326061,0.206252,0.289008
2,0.0,0.781569,0.442103,0.661957,0.553572,0.645458
3,0.0,0.810585,0.485475,0.637629,0.611490,0.713029
4,0.0,0.226851,0.267024,0.535453,0.604695,0.228915


In [10]:
# compute scaler on train
mean = X_train.mean(axis=0, keepdims=True)
std = X_train.std(axis=0, keepdims=True)
std[std == 0.0] = 1.0  # tránh chia cho 0

# apply scaler
X_train_n = (X_train - mean) / std
X_val_n   = (X_val   - mean) / std
X_test_n  = (X_test  - mean) / std

# to tensors
tX_train = torch.from_numpy(X_train_n)
ty_train = torch.from_numpy(y_train)
tX_val   = torch.from_numpy(X_val_n)
ty_val   = torch.from_numpy(y_val)
tX_test  = torch.from_numpy(X_test_n)
ty_test  = torch.from_numpy(y_test)

train_ds = TensorDataset(tX_train, ty_train)
val_ds   = TensorDataset(tX_val, ty_val)
test_ds  = TensorDataset(tX_test, ty_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# lưu scaler nếu cần
scaler = {"mean": mean, "std": std}

In [11]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_sizes=[64,32], dropout=0.1):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))  # regression output
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# instantiate
input_dim = tX_train.shape[1]
model = MLP(input_dim, hidden_sizes=HIDDEN, dropout=DROPOUT).to(DEVICE)
print(model)

MLP(
  (net): Sequential(
    (0): Linear(in_features=5, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [12]:
criterion = nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=LR)

def train_epoch(model, loader, opt, criterion, device):
    model.train()
    running_loss = 0.0
    n = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        pred = model(xb)
        loss = criterion(pred, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        bs = xb.size(0)
        running_loss += loss.item() * bs
        n += bs
    return running_loss / n

def eval_loss(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    n = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            bs = xb.size(0)
            running_loss += loss.item() * bs
            n += bs
    return running_loss / n

# training loop
best_val = float("inf")
best_state = None
history = {"train_loss": [], "val_loss": []}

for epoch in range(1, EPOCHS+1):
    tr_loss = train_epoch(model, train_loader, opt, criterion, DEVICE)
    val_loss = eval_loss(model, val_loader, criterion, DEVICE)
    history["train_loss"].append(tr_loss)
    history["val_loss"].append(val_loss)
    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.cpu() for k, v in model.state_dict().items()}
    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d}  train_loss={tr_loss:.6f}  val_loss={val_loss:.6f}")

# load best
if best_state is not None:
    model.load_state_dict(best_state)
print("Best val loss:", best_val)

Epoch 001  train_loss=0.192170  val_loss=0.098240
Epoch 010  train_loss=0.007382  val_loss=0.002349
Epoch 020  train_loss=0.005466  val_loss=0.001109
Epoch 030  train_loss=0.004441  val_loss=0.000420
Epoch 040  train_loss=0.003449  val_loss=0.000446
Epoch 050  train_loss=0.003629  val_loss=0.000440
Epoch 060  train_loss=0.002846  val_loss=0.000204
Epoch 070  train_loss=0.002772  val_loss=0.000231
Epoch 080  train_loss=0.002487  val_loss=0.000311
Epoch 090  train_loss=0.002370  val_loss=0.000355
Epoch 100  train_loss=0.002471  val_loss=0.000125
Best val loss: 0.0001252193032996729


In [13]:
def evaluate_full(model, loader, device):
    model.eval()
    ys = []
    preds = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            pred = model(xb).cpu().numpy().reshape(-1)
            preds.append(pred)
            ys.append(yb.numpy().reshape(-1))
    y_true = np.concatenate(ys)
    y_pred = np.concatenate(preds)
    mse = np.mean((y_true - y_pred) ** 2)
    mae = np.mean(np.abs(y_true - y_pred))
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot != 0 else float("nan")
    return {"mse": mse, "mae": mae, "r2": r2}, y_true, y_pred

results, y_true, y_pred = evaluate_full(model, test_loader, DEVICE)
print("Test results:", results)

Test results: {'mse': np.float32(0.00012759579), 'mae': np.float32(0.008455854), 'r2': np.float32(0.9928832)}
